# Notebook 5: Bracket Generator
## Generate optimal brackets for multiple pool configurations
---
**Input:** Simulation results from `results/simulation_results.csv`  
**Output:** `brackets/` directory with one CSV per pool + printable summaries

Run this AFTER Notebooks 1–3. Re-run whenever you update simulations.

In [1]:
# ============================================================
# 5.0 CONFIG & IMPORTS
# ============================================================
import os, json, warnings, logging
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("brackets")

RESULTS_DIR = Path("./results")
BRACKETS_DIR = Path("./brackets")
BRACKETS_DIR.mkdir(exist_ok=True)

sim_results = pd.read_csv(RESULTS_DIR / "simulation_results.csv")
log.info(f"Loaded simulation results: {sim_results.shape[0]} teams")

# ==========================================================
# POOL DEFINITIONS
# ==========================================================
# Entry fee determines whether to maximize P(winning) or E(profit).
# Small free pools: maximize P(1st place) = pick chalk.
# Large paid pools: maximize E[payout - entry_fee] = need leverage.
# Multiple free pools: VARY picks across pools to cover more outcomes.

POOLS = {
    "Family": {
        "scoring": "espn", "pool_size": 10,
        "espn_points": [10, 20, 40, 80, 160, 320],
        "entry_fee": 0, "payout_1st": 20,
        "description": "ESPN default, ~10 people, free",
    },
    "Fantasy_Prem": {
        "scoring": "espn", "pool_size": 4,
        "espn_points": [10, 20, 40, 80, 160, 320],
        "entry_fee": 10, "payout_1st": 40,
        "description": "ESPN default, ~4 people, $10 entry",
    },
    "College": {
        "scoring": "yahoo", "pool_size": 8,
        "yahoo_points": [1, 2, 4, 8, 16, 32],
        "entry_fee": 10, "payout_1st": 80,
        "description": "Yahoo default, ~8 people, $10 entry",
    },
    "Dereks": {
        "scoring": "espn", "pool_size": 20,
        "espn_points": [10, 20, 40, 80, 160, 320],
        "entry_fee": 5, "payout_1st": 75,
        "description": "ESPN default, ~20 people, $5 pot",
    },
    "DK_Company": {
        "scoring": "espn", "pool_size": 2500,
        "espn_points": [10, 20, 40, 80, 160, 320],
        "entry_fee": 0, "payout_1st": 250,
        "description": "ESPN default, ~2500 people",
    },
    "DK_ATLAS_Org": {
        "scoring": "custom_inverse", "pool_size": 35,
        "entry_fee": 10, "payout_1st": 300,
        "description": "1pt per correct / # people with that pick, ~35 people",
    },
    "DK_Analytics_Org": {
        "scoring": "espn", "pool_size": 100,
        "espn_points": [10, 20, 40, 80, 160, 320],
        "entry_fee": 0, "payout_1st": 250,
        "description": "ESPN default, ~100 people",
    },
}

print(f"Configured {len(POOLS)} pools:")
for name, cfg in POOLS.items():
    print(f"  {name}: {cfg['description']}")

2026-03-18 23:36:57,423 [INFO] Loaded simulation results: 64 teams


Configured 7 pools:
  Family: ESPN default, ~10 people, free
  Fantasy_Prem: ESPN default, ~4 people, $10 entry
  College: Yahoo default, ~8 people, $10 entry
  Dereks: ESPN default, ~20 people, $5 pot
  DK_Company: ESPN default, ~2500 people
  DK_ATLAS_Org: 1pt per correct / # people with that pick, ~35 people
  DK_Analytics_Org: ESPN default, ~100 people


In [2]:
# ============================================================
# 5.1 CONTRARIAN & OWNERSHIP MODEL
# ============================================================

# Historical base rates for grounding
# All 4 1-seeds in F4: 9% of seasons (2/23). Three+ 1-seeds: 13%.
# 3-seed champion: 13% (3/23). 1-seed champion: 70%.
# So all-chalk F4 is a ~9% event — unlikely but highest-EV for tiny pools.

def get_contrarian_factor(pool_size):
    """Contrarian aggressiveness based on pool size.
    Capped at 0.40 — even in huge pools, champion is seed 1-2 ~80% of the time.
    """
    if pool_size < 10:  return 0.00
    if pool_size < 25:  return 0.05
    if pool_size < 75:  return 0.10
    if pool_size < 200: return 0.20
    if pool_size < 1000: return 0.30
    return 0.40


def estimate_public_ownership(seed, round_name):
    """Estimate what % of a typical pool picks this seed to advance."""
    base = {
        1: {"R64": .99, "R32": .93, "S16": .80, "E8": .58, "F4": .38, "Championship": .20, "Champion": .12},
        2: {"R64": .94, "R32": .82, "S16": .58, "E8": .33, "F4": .17, "Championship": .09, "Champion": .05},
        3: {"R64": .85, "R32": .62, "S16": .38, "E8": .18, "F4": .08, "Championship": .04, "Champion": .015},
        4: {"R64": .79, "R32": .52, "S16": .28, "E8": .12, "F4": .05, "Championship": .025, "Champion": .008},
        5: {"R64": .65, "R32": .38, "S16": .18, "E8": .07, "F4": .03, "Championship": .012, "Champion": .004},
        6: {"R64": .63, "R32": .35, "S16": .14, "E8": .05, "F4": .02, "Championship": .008, "Champion": .003},
        7: {"R64": .61, "R32": .30, "S16": .12, "E8": .04, "F4": .015, "Championship": .006, "Champion": .002},
        8: {"R64": .51, "R32": .22, "S16": .08, "E8": .025, "F4": .008, "Championship": .003, "Champion": .001},
    }
    if seed <= 8:
        return base.get(seed, {}).get(round_name, 0.005)
    s8 = base.get(8, {}).get(round_name, 0.005)
    return max(0.001, s8 * (0.5 ** ((seed - 8) / 4)))


def optimize_bracket(sim_results, pool_config, pool_name, champion_override=None):
    """Generate optimal bracket picks for a specific pool.
    
    champion_override: force a specific champion team name (for portfolio diversification)
    """
    scoring = pool_config["scoring"]
    cf = get_contrarian_factor(pool_config["pool_size"])
    
    log.info(f"  {pool_name}: scoring={scoring}, pool={pool_config['pool_size']}, cf={cf:.2f}")
    
    df = sim_results.copy()
    round_names = ["R64", "R32", "S16", "E8", "F4", "Championship", "Champion"]
    
    # Probability floors by round (don't recommend negligible-probability picks)
    prob_floor = {"R64": 0.0, "R32": 0.0, "S16": 0.05, "E8": 0.03,
                  "F4": 0.10, "Championship": 0.02, "Champion": 0.02}
    
    for rnd in round_names:
        prob_col = f"prob_{rnd}"
        if prob_col not in df.columns:
            continue
        
        df[f"own_{rnd}"] = df["seed"].apply(lambda s: estimate_public_ownership(s, rnd))
        our_prob = df[prob_col]
        ownership = df[f"own_{rnd}"].clip(lower=0.001)
        floor = prob_floor.get(rnd, 0.0)
        eligible = our_prob >= floor
        
        if scoring == "custom_inverse":
            df[f"value_{rnd}"] = np.where(eligible, our_prob / ownership, 0)
        else:
            raw_value = (1 - cf) * our_prob + cf * (our_prob / ownership)
            df[f"value_{rnd}"] = np.where(eligible, raw_value, our_prob)
    
    # Apply champion override if specified
    if champion_override:
        df.loc[df["team"] == champion_override, "value_Champion"] = 999
    
    return df.sort_values("value_Champion", ascending=False), cf


print("Optimization functions loaded")

Optimization functions loaded


In [3]:
# ============================================================
# 5.2 GENERATE ALL BRACKETS
# ============================================================
# STRATEGY: For small pools (<15 people), you're entering multiple brackets.
# Don't submit the same bracket everywhere — diversify your champion picks
# to cover more outcomes. The "portfolio" approach:
#   - Pool 1: pick the model's #1 champion
#   - Pool 2: pick the #2 champion (different team)
#   - Pool 3: pick the #3 champion
#   - etc.
# This way if ANY of your champions hits, you win one pool.

# First, rank champion candidates for small-pool diversification
champ_ranking = sim_results.nlargest(6, "prob_Champion")[["team", "seed", "region", "prob_Champion"]]
print("Champion candidates (for portfolio diversification):")
for i, (_, row) in enumerate(champ_ranking.iterrows()):
    print(f"  #{i+1}: {row['team']} ({int(row['seed'])}-seed, {row['prob_Champion']:.1%})")

# Assign champion picks to small pools for diversification
small_pools = ["Family", "Fantasy_Prem", "College", "Dereks"]
champ_assignments = {}
champ_list = champ_ranking["team"].tolist()
for i, pool_name in enumerate(small_pools):
    champ_assignments[pool_name] = champ_list[i] if i < len(champ_list) else None
    
# Large pools use the optimizer's value-based pick
large_pools = ["DK_Company", "DK_ATLAS_Org", "DK_Analytics_Org"]

all_brackets = {}

for pool_name, pool_config in POOLS.items():
    override = champ_assignments.get(pool_name)
    bracket_df, cf = optimize_bracket(sim_results, pool_config, pool_name, champion_override=override)
    all_brackets[pool_name] = bracket_df
    bracket_df.to_csv(BRACKETS_DIR / f"bracket_{pool_name}.csv", index=False)
    
    print(f"\n{'='*65}")
    print(f"  {pool_name}: {pool_config['description']}")
    print(f"  Contrarian factor: {cf:.2f}")
    if override:
        print(f"  Champion override: {override} (portfolio diversification)")
    print(f"{'='*65}")
    
    # Champion
    champ_contenders = bracket_df[bracket_df["prob_Champion"] >= 0.02]
    if len(champ_contenders) == 0:
        champ_contenders = bracket_df.nlargest(5, "prob_Champion")
    champ = champ_contenders.sort_values("value_Champion", ascending=False).iloc[0]
    print(f"  CHAMPION: {champ['team']} ({int(champ['seed'])}-seed)")
    print(f"     Win prob: {champ['prob_Champion']:.1%}, Value: {champ['value_Champion']:.4f}")
    
    # Final Four
    print(f"  FINAL FOUR:")
    for region in ["East", "West", "South", "Midwest"]:
        region_df = bracket_df[bracket_df["region"] == region]
        contenders = region_df[region_df["prob_F4"] >= 0.10]
        if len(contenders) == 0:
            contenders = region_df.nlargest(3, "prob_F4")
        if "value_F4" in contenders.columns:
            top = contenders.sort_values("value_F4", ascending=False).iloc[0]
        else:
            top = contenders.sort_values("prob_F4", ascending=False).iloc[0]
        print(f"     {region:8s}: {top['team']:20s} ({int(top['seed'])}-seed, "
              f"F4 prob={top['prob_F4']:.1%})")
    
    if cf > 0.1:
        print(f"  UPSET PICKS (high leverage):")
        upsets = bracket_df[
            (bracket_df["seed"] >= 5) & (bracket_df["prob_S16"] > 0.1)
        ].sort_values("value_S16" if "value_S16" in bracket_df.columns else "prob_S16",
                       ascending=False).head(5)
        for _, u in upsets.iterrows():
            print(f"     {u['team']:20s} ({int(u['seed'])}-seed) S16: {u['prob_S16']:.1%}")

log.info(f"\nAll brackets saved to {BRACKETS_DIR}/")

2026-03-18 23:36:57,433 [INFO]   Family: scoring=espn, pool=10, cf=0.05
2026-03-18 23:36:57,439 [INFO]   Fantasy_Prem: scoring=espn, pool=4, cf=0.00
2026-03-18 23:36:57,444 [INFO]   College: scoring=yahoo, pool=8, cf=0.00
2026-03-18 23:36:57,449 [INFO]   Dereks: scoring=espn, pool=20, cf=0.05
2026-03-18 23:36:57,454 [INFO]   DK_Company: scoring=espn, pool=2500, cf=0.40
2026-03-18 23:36:57,460 [INFO]   DK_ATLAS_Org: scoring=custom_inverse, pool=35, cf=0.10
2026-03-18 23:36:57,465 [INFO]   DK_Analytics_Org: scoring=espn, pool=100, cf=0.20
2026-03-18 23:36:57,471 [INFO] 
All brackets saved to brackets/


Champion candidates (for portfolio diversification):
  #1: Michigan (1-seed, 25.4%)
  #2: Duke (1-seed, 21.6%)
  #3: Arizona (1-seed, 17.1%)
  #4: Florida (1-seed, 9.1%)
  #5: Purdue (2-seed, 6.5%)
  #6: Houston (2-seed, 5.7%)

  Family: ESPN default, ~10 people, free
  Contrarian factor: 0.05
  Champion override: Michigan (portfolio diversification)
  CHAMPION: Michigan (1-seed)
     Win prob: 25.4%, Value: 999.0000
  FINAL FOUR:
     East    : Duke                 (1-seed, F4 prob=64.7%)
     West    : Arizona              (1-seed, F4 prob=55.0%)
     South   : Florida              (1-seed, F4 prob=38.3%)
     Midwest : Michigan             (1-seed, F4 prob=59.9%)

  Fantasy_Prem: ESPN default, ~4 people, $10 entry
  Contrarian factor: 0.00
  Champion override: Duke (portfolio diversification)
  CHAMPION: Duke (1-seed)
     Win prob: 21.6%, Value: 999.0000
  FINAL FOUR:
     East    : Duke                 (1-seed, F4 prob=64.7%)
     West    : Arizona              (1-seed, F4 prob=55

In [4]:
# ============================================================
# 5.3 COMPARISON TABLE
# ============================================================
# Side-by-side champion + Final Four picks across all pools

comparison = []
for pool_name, bracket_df in all_brackets.items():
    champ = bracket_df.iloc[0]
    row = {
        "Pool": pool_name,
        "Champion": f"{champ['team']} ({int(champ['seed'])})",
        "Champ_Prob": f"{champ['prob_Champion']:.1%}",
    }
    for region in ["East", "West", "South", "Midwest"]:
        region_df = bracket_df[bracket_df["region"] == region]
        val_col = "value_F4" if "value_F4" in region_df.columns else "prob_F4"
        top = region_df.sort_values(val_col, ascending=False).iloc[0]
        row[f"F4_{region}"] = f"{top['team']} ({int(top['seed'])})"
    comparison.append(row)

comp_df = pd.DataFrame(comparison)
print("\n  BRACKET COMPARISON ACROSS POOLS:")
print("=" * 100)
display(comp_df)
comp_df.to_csv(BRACKETS_DIR / "comparison_summary.csv", index=False)

print(f"\n All {len(POOLS)} brackets generated in {BRACKETS_DIR}/")


  BRACKET COMPARISON ACROSS POOLS:


,Pool,Champion,Champ_Prob,F4_East,F4_West,F4_South,F4_Midwest
0,Family,Michigan (1),25.4%,Duke (1),Arizona (1),Florida (1),Michigan (1)
1,Fantasy_Prem,Duke (1),21.6%,Duke (1),Arizona (1),Florida (1),Michigan (1)
2,College,Arizona (1),17.1%,Duke (1),Arizona (1),Florida (1),Michigan (1)
3,Dereks,Florida (1),9.1%,Duke (1),Arizona (1),Florida (1),Michigan (1)
4,DK_Company,Illinois (3),4.8%,Duke (1),Arizona (1),Illinois (3),Michigan (1)
5,DK_ATLAS_Org,Illinois (3),4.8%,Duke (1),Purdue (2),Illinois (3),Michigan (1)
6,DK_Analytics_Org,Illinois (3),4.8%,Duke (1),Arizona (1),Illinois (3),Michigan (1)



 All 7 brackets generated in brackets/


## Bracket Generator Summary

Each pool gets a tailored bracket based on:
- **Scoring system** (ESPN, Yahoo, custom inverse ownership)
- **Pool size** → contrarian factor (small pools play chalk, large pools differentiate)
- **Ownership estimates** → leverage (pick teams the public undervalues)

### Pool Strategy Guide:
| Pool | Strategy | Why |
|------|----------|-----|
| Family (8-12) | Pure chalk | Small pool, just need to be right |
| Fantasy Prem (3-5) | Pure chalk | Tiny pool, pick the best team |
| College (6-10) | Chalk | Small pool, just need to be right |
| Derek's (12-15) | Chalk | Still small enough for chalk |
| DK Company (2k+) | Max contrarian | Need differentiation to win |
| DK ATLAS (20-50) | Built-in contrarian | Scoring = 1/ownership, naturally rewards uniqueness |
| DK Analytics (75-100) | Moderate contrarian | Large enough to need some differentiation |